# Baseline real-only đủ 5 task — EXP-B1 / EXP-B2

Ngày tạo: 10/10/2026, UTC+7. Real run trên RTX 4090, protocol 100+4×25, release/holdout đã khóa. Train lại Task 1 từ public pretrained với budget đã chọn, rồi Task 2–5 từ task_final trước đó; không có replay loader. ResNet và ConvNeXt cùng data/resolution/effective batch/epochs/evaluator, không bật LoRA/K=3/synthetic ablations.

Mọi artifacts dưới data. [Runbook](../../docs/project/TRAINING_NOTEBOOK_RUNBOOK.md) hướng dẫn tmux, logs/resume, đường upload và cấu hình. Mặc định dry run CPU/6 labels qua đủ 5 tasks, không tạo kết quả nghiên cứu.


## 1. Budget được chọn từ pilot val
Pilot và transition phải hoàn tất cùng campaign. Real run yêu cầu baseline_epochs là số nguyên dương trong settings; mặc định null để người dùng chọn từ pilot val trước training dài. Dry run dùng 1 epoch/task để kiểm chứng lifecycle. Giữ batch/epochs/data/resolution khi resume; thay budget sau task_final cần campaign mới.


In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout.io import load_json
from autocheckout.training_workflow import prepare, run_phase, resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'configs/training/notebooks.json'))
SETTINGS = load_json(SETTINGS_PATH)
PHASE = 'baseline'
PLAN = prepare(SETTINGS, PHASE)
print(json.dumps({k: PLAN[k] for k in ['phase_dir', 'dry_run', 'epochs', 'start_task', 'n_tasks', 'resolution', 'batch_size', 'effective_batch']}, indent=2))


## 2. Chạy và resume toàn protocol
Mỗi experiment chạy tuần tự task 1–5. Task đã có checkpoint final, hai predictions và snapshot metrics được bỏ qua; thiếu prediction chỉ predict-only; task đang dở resume từ last.ckpt. Logger ghi một dòng mỗi optimizer step, progress JSON và checkpoint receipts. Run ngắt bất kỳ thời điểm nào giữ artifact, chạy lại cùng settings để tiếp tục từ checkpoint đã lưu.

Ngay sau mỗi task, evaluator xuất mAP@C/P/A, matrix/forgetting, counting old/new/levels và policy val cho các stages đã hoàn tất. Snapshot nằm ở stage_metrics/task_<t>/; calibration_count_through_task_<t>.json giữ riêng policy cho từng prefix. Metrics đầy đủ mới nhất nằm ở metrics_count_test.json / metrics_cl_test.json, oracle mặc định tắt.


In [ ]:
SUMMARY = run_phase(PLAN)
for case in SUMMARY['cases']:
    print(case['experiment'])
    print(json.dumps([{'task': t['task_id'], 'optimizer_steps': t['optimizer_steps'], 'seconds': t['seconds'], 'peak_gpu_gib': t['peak_gpu_gib'], 'missing_prototypes': t['prototype_diagnostics']} for t in case['tasks']], indent=2))
    print('Forgetting:', case['detection']['forgetting'])
    print('Counting stages:', sorted(case['counting']))
assert SUMMARY['input_hashes_unchanged']
print('Status:', SUMMARY['status'], 'dry_run:', SUMMARY['dry_run'], 'research_metrics:', SUMMARY['research_metrics'])
print('Summary:', Path(PLAN['phase_dir']) / 'summary.json')


## 3. Lưu kết quả và tiếp tục
Giữ cả campaign folder dưới data: checkpoint, predictions, policies, config/hash receipts, CSV/JSONL/logs và các bản notebook đã execute. tmux bảo vệ khi SSH mất kết nối, không bảo vệ khi instance/disk bị xóa. Lấy data về storage bền vững trước khi kết thúc instance.

COMPLETED dry run chỉ chứng minh notebook/runner/resume/teacher/evaluation hoạt động với model nhỏ. Real baseline cần xem loss/val, missing prototype coverage và metrics từng stage trước khi kết luận convergence hay so sánh backbone. LoRA, K=3 và synthetic triển khai sau trong campaign riêng.
